# LoRa Chirp Spread Spectrum

This notebook introduces LoRa-style chirp spread spectrum using synthetic chirps. It focuses on the visual intuition: spreading factor changes chirp duration, symbol rate, and the time-bandwidth tradeoff.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## Chirps as Symbols

Instead of holding a fixed carrier state for each symbol, LoRa sweeps frequency across the channel. That makes the signal look like a diagonal trace in a spectrogram.

In [ ]:
def lora_like_chirp(sf=7, bw=125_000, fs=500_000):
    symbol_time = (2 ** sf) / bw
    t = np.arange(0, symbol_time, 1 / fs)
    chirp = signal.chirp(t, f0=-bw / 2, f1=bw / 2, t1=symbol_time, method="linear")
    return t, chirp

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
audio_out = audio_output_widget()

def update_lora(sf=7, bw=125_000):
    t, chirp = lora_like_chirp(sf=int(sf), bw=bw)
    axes[0].clear()
    axes[1].clear()
    plot_waveform(chirp[:3000], fs=500_000, ax=axes[0], title=f"Chirp waveform, SF{sf}")
    plot_spectrogram(chirp, fs=500_000, ax=axes[1], title="Chirp spectrogram", nperseg=512, noverlap=384)
    axes[1].set_ylim(0, bw / 2)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, resample_signal(chirp, 500_000, 44_100), rate=44_100)

controls = widgets.interactive(
    update_lora,
    sf=int_slider(min_value=7, max_value=12, step=1, value=7, description="SF"),
    bw=float_slider(min_value=62_500, max_value=500_000, step=62_500, value=125_000, description="BW", readout_format=".0f"),
)
display(controls, audio_out)


## Symbol Time vs Spreading Factor

Increasing spreading factor makes the chirp longer, which improves sensitivity but lowers data rate.

In [ ]:
out = widgets.Output()

def update_tradeoff(sf=7, bw=125_000):
    symbol_time = (2 ** sf) / bw
    raw_symbol_rate = 1 / symbol_time
    with out:
        out.clear_output(wait=True)
        print(f"Spreading factor: SF{sf}")
        print(f"Bandwidth: {bw/1000:.1f} kHz")
        print(f"Symbol time: {symbol_time * 1e3:.3f} ms")
        print(f"Symbol rate: {raw_symbol_rate:.2f} symbols/s")

controls = widgets.interactive(
    update_tradeoff,
    sf=int_slider(min_value=7, max_value=12, step=1, value=7, description="SF"),
    bw=float_slider(min_value=62_500, max_value=500_000, step=62_500, value=125_000, description="BW", readout_format=".0f"),
)
display(controls, out)


## Key Takeaway

LoRa's chirps trade speed for resilience. The longer the chirp relative to the bandwidth, the easier it is to dig weak energy out of noise, but the fewer symbols you send per second.